# putEMG — Within-Subject Feature-Based Evaluation

Mirrors the within-subject **deep learning** protocol from `experiments/within_subject/deep_learning/`
but operates on **hand-crafted features** (libemg, 50 feature groups × 24 channels = 3192-dim per window).
Each rep is converted to a `(26, 3192)` temporal sequence and classified by a sequence model.

| # | Model | Description |
|---|-------|-------------|
| 1 | **FeatureLSTM** | Bidirectional LSTM over the 26-step feature sequence |
| 2 | **FeatureGRU** | Bidirectional GRU — lighter alternative to LSTM |
| 3 | **FeatureTransformer** | Self-attention encoder with sinusoidal positional encoding |

**Within-subject split (per fold):**
- Each subject's reps are split into `N_FOLDS` stratified folds
- **Train** — 2 folds (~187 reps) | **Test** — 1 fold (~93 reps)
- Rotate `N_FOLDS` times per subject; report per-subject mean → grand mean ± std
- Early stopping monitors **training loss** (no dev set — per-subject data is limited)

**Benchmark:** putEMG paper SVM/RMS ~90% (same within-subject protocol)

> **Prerequisites**: Run `data_preprocessing/driver.ipynb` first to generate per-subject `.mat` files.
> Features are extracted automatically by this notebook on first run.

In [ ]:
# -- Main libraries --
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader


# -- Shared utilities (src/) --
_SRC = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', 'src'))
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)

# -- Created functions --
import feature_based_models
from emg_loader import (
    BCIDataset,
    load_all_subjects,
    load_feature_subjects,
    make_within_subject_loaders,
)
from feature_extraction import batch_extract


# -- Config --
DATA_DIR    = '/Volumes/KRIS/data/UG_per_subject'         # raw preprocessed .mat files
FEATURE_DIR = '/Volumes/KRIS/data/features_sequence'      # shared feature cache (.npz files)
MODE        = 'sequence'
BATCH_SIZE  = 16

---
## Feature Extraction

If `FEATURE_DIR` is empty or incomplete, extract features from the raw `.mat` files and cache them as `.npz`.
Subjects with existing files are skipped by `batch_extract` — safe to re-run.

In [ ]:
os.makedirs(FEATURE_DIR, exist_ok=True)
expected = len([f for f in os.listdir(DATA_DIR) if f.endswith('.mat')])
present  = len([f for f in os.listdir(FEATURE_DIR) if f.endswith(f'_{MODE}.npz')])
print(f'Raw subjects     : {expected}')
print(f'Cached features  : {present}  ({FEATURE_DIR})')

if present < expected:
    print(f'\nExtracting features for {expected - present} subject(s)\u2026')
    raw_subjects = load_all_subjects(DATA_DIR)
    batch_extract(raw_subjects, output_dir=FEATURE_DIR, mode=MODE)
else:
    print('All feature files present — skipping extraction.')

subjects = load_feature_subjects(FEATURE_DIR, mode=MODE)

In [ ]:
# ── Training & evaluation utilities ────────────────────────────────────
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def train(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out  = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)


def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            pred = model(X).argmax(dim=1)
            correct += (pred == y).sum().item()
            total   += y.size(0)
    return correct / total


def evaluateFinal(model, test_loader, device):
    model.eval()
    correct, total = 0, 0
    actual    = torch.tensor([], dtype=torch.int64)
    predicted = torch.tensor([], dtype=torch.int64)
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            pred = model(X).argmax(dim=1)
            correct   += (pred == y).sum().item()
            total     += y.size(0)
            actual    = torch.cat((actual,    y.cpu()),    dim=0)
            predicted = torch.cat((predicted, pred.cpu()), dim=0)

        print(np.unique(predicted.numpy(), return_counts=True))
        cm = confusion_matrix(actual.numpy(), predicted.numpy())
        ConfusionMatrixDisplay(cm).plot()

    print(f'The accuracy for the test is {(correct / total) * 100:.2f}%')
    return correct / total

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# -- Config --
N_FOLDS     = 3       # folds per subject — matches putEMG paper protocol
N_SUBJECTS  = None    # None = all
SEED        = 42      # random seed for stratified fold generation
PATIENCE    = 5       # early-stopping patience (epochs without improvement)
MAX_EPOCHS  = 30
MIN_DELTA   = 0.001   # minimum improvement to reset patience
DROPOUT     = 0.3     # sequence models overfit faster on 3192-dim features
LR          = 1e-3
WEIGHTS_DIR = 'weights'
RESULTS_DIR = 'results'

---
## Training

Each model is trained independently **per subject** with:
- **Stratified 3-fold CV** — class balance preserved across every fold
- **Early stopping** — stops when training loss doesn't improve by `MIN_DELTA` for `PATIENCE` epochs
- **ReduceLROnPlateau** — halves the LR when training loss plateaus for 3 epochs
- **Best-state checkpointing** — restores the lowest-loss weights before test evaluation

Results are collected per fold. After all folds, mean accuracy is reported per subject → grand mean ± std across all subjects.

In [ ]:
model_registry = {
    'FeatureLSTM':        lambda: feature_based_models.FeatureLSTM(dropout_rate=DROPOUT),
    'FeatureGRU':         lambda: feature_based_models.FeatureGRU(dropout_rate=DROPOUT),
    'FeatureTransformer': lambda: feature_based_models.FeatureTransformer(dropout_rate=DROPOUT),
}

In [ ]:
subject_results = {}          # {subject_name: {model_name: [fold accs]}}

run_subjects = subjects[:N_SUBJECTS] if N_SUBJECTS else subjects
print(f'Subjects : {len(run_subjects)}')
print(f'Folds    : {N_FOLDS} per subject')
print(f'Models   : {list(model_registry.keys())}')
print(f'Total    : {len(run_subjects) * len(model_registry)} subject-model pairs')

In [ ]:
for sub_idx, (sub_name, X_sub, y_sub) in enumerate(run_subjects):

    sub_id = sub_name.split('_')[2]                     # 'features_subject_03_sequence.npz' → '03'

    print(f"\n{'#'*60}")
    print(f"  Subject {sub_idx + 1}/{len(run_subjects)}  —  subject_{sub_id}")
    print(f"{'#'*60}")

    subject_results.setdefault(sub_name, {})

    for name, build_fn in model_registry.items():

        model_dir = os.path.join(WEIGHTS_DIR, name)
        save_path = os.path.join(model_dir, f'subject_{sub_id}.pt')

        # -- Skip if already trained --
        if os.path.exists(save_path):
            checkpoint = torch.load(save_path, map_location='cpu')
            subject_results[sub_name][name] = checkpoint['fold_accs']
            print(f"  [{name}] subject_{sub_id} already trained "
                  f"(mean={np.mean(checkpoint['fold_accs'])*100:.2f}%) — skipping.")
            continue

        print(f"\n{'='*50}")
        print(f"  Training: {name}")
        print(f"{'='*50}")

        fold_accs   = []
        fold_states = []

        for fold_idx in range(N_FOLDS):

            train_loader, test_loader = make_within_subject_loaders(
                X_sub, y_sub,
                n_folds=N_FOLDS, test_fold_idx=fold_idx,
                batch_size=BATCH_SIZE, seed=SEED,
            )

            model     = build_fn().to(device)
            optimizer = optim.Adam(model.parameters(), lr=LR)
            scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5,
                                          patience=3, min_lr=1e-6)
            criterion = nn.CrossEntropyLoss()

            best_loss  = float('inf')
            best_state = None
            bad_epochs = 0

            for epoch in range(MAX_EPOCHS):
                tr_loss = train(model, train_loader, criterion, optimizer, device)
                curr_lr = optimizer.param_groups[0]['lr']

                if tr_loss < best_loss - MIN_DELTA:
                    best_loss  = tr_loss
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                    bad_epochs = 0
                else:
                    bad_epochs += 1

                scheduler.step(tr_loss)

                print(f"  Epoch {epoch+1:3d}: loss={tr_loss:.4f}  "
                      f"best={best_loss:.4f}  lr={curr_lr:.2e}")

                if bad_epochs >= PATIENCE:
                    print(f"  Early stopping at epoch {epoch+1}.")
                    break

            model.load_state_dict(best_state)

            print(f"\n  --- {name} | subject_{sub_id}, Fold {fold_idx + 1} — Test Evaluation ---")
            test_acc = evaluate(model, test_loader, device)
            print(f"  Test accuracy: {test_acc * 100:.2f}%")

            fold_accs.append(test_acc)
            fold_states.append(best_state)

        # -- Save best-fold weights for this subject-model pair --
        best_fold_idx = int(np.argmax(fold_accs))
        subject_results[sub_name][name] = fold_accs

        os.makedirs(model_dir, exist_ok=True)
        torch.save({
            'model_name':   name,
            'subject_id':   f'subject_{sub_id}',
            'subject_file': sub_name,
            'fold_accs':    fold_accs,
            'mean_acc':     float(np.mean(fold_accs)),
            'best_fold':    best_fold_idx,
            'n_folds':      N_FOLDS,
            'dropout':      DROPOUT,
            'state_dict':   fold_states[best_fold_idx],
        }, save_path)

        print(f"\n  Saved → {save_path}")
        print(f"  Fold accs : {[f'{a*100:.2f}%' for a in fold_accs]}")
        print(f"  Mean acc  : {np.mean(fold_accs)*100:.2f}%")

---
## Results Summary

In [ ]:
print(f"\n{'Model':<22} {'Subjects':>8}  {'Mean Acc':>10}  {'Std':>8}")
print('-' * 55)
for name in model_registry:
    sub_means = [
        np.mean(subject_results[s][name]) * 100
        for s in subject_results
        if name in subject_results[s] and subject_results[s][name]
    ]
    if sub_means:
        mean, std = np.mean(sub_means), np.std(sub_means)
        print(f"{name:<22} {len(sub_means):>8}  {mean:>9.2f}%  {std:>7.2f}%")
    else:
        print(f"{name:<22} {'\u2014':>8}  {'\u2014':>10}  {'\u2014':>8}")

# Bar chart — grand mean accuracy per model
names_with_results = [
    n for n in model_registry
    if any(n in subject_results[s] and subject_results[s][n] for s in subject_results)
]
means = [
    np.mean([np.mean(subject_results[s][n]) * 100 for s in subject_results if n in subject_results[s] and subject_results[s][n]])
    for n in names_with_results
]
stds = [
    np.std([np.mean(subject_results[s][n]) * 100 for s in subject_results if n in subject_results[s] and subject_results[s][n]])
    for n in names_with_results
]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(names_with_results, means, yerr=stds, capsize=5, color='steelblue')
ax.axhline(y=90, color='red', linestyle='--', alpha=0.6, label='putEMG paper (~90%)')
ax.set_ylim(0, 110)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title(f'Within-Subject Feature-Based Results ({N_FOLDS} fold(s) × {len(subject_results)} subject(s)) — Mean ± Std')
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f'{m:.1f}%', ha='center', va='bottom', fontsize=10)
ax.legend()
plt.tight_layout()
plt.show()

---
## Best Architecture

In [ ]:
# Weights were already saved per subject-model pair during training.
# This cell identifies the winning architecture by grand mean across subjects.

model_means = {}
for name in model_registry:
    sub_means = [
        np.mean(subject_results[s][name]) * 100
        for s in subject_results
        if name in subject_results[s] and subject_results[s][name]
    ]
    if sub_means:
        model_means[name] = (np.mean(sub_means), np.std(sub_means), len(sub_means))

print(f"\n{'Model':<22} {'Subjects':>8}  {'Mean Acc':>10}  {'Std':>8}")
print('-' * 55)
for name, (mean, std, n) in model_means.items():
    print(f"{name:<22} {n:>8}  {mean:>9.2f}%  {std:>7.2f}%")

if model_means:
    best_architecture = max(model_means, key=lambda n: model_means[n][0])
    best_mean         = model_means[best_architecture][0]
    print(f"\n→ Best architecture : {best_architecture}  ({best_mean:.2f}%)")
    print(f"  Weights saved in  : {os.path.join(WEIGHTS_DIR, best_architecture)}/")

---
## Save Report

In [ ]:
from datetime import datetime

os.makedirs(RESULTS_DIR, exist_ok=True)
report_path = os.path.join(RESULTS_DIR, 'report.txt')

# ── Compute per-model grand stats ────────────────────────────────────
model_stats = {}
for name in model_registry:
    sub_means = [
        np.mean(subject_results[s][name]) * 100
        for s in subject_results
        if name in subject_results[s] and subject_results[s][name]
    ]
    if sub_means:
        model_stats[name] = (np.mean(sub_means), np.std(sub_means), len(sub_means))

best_architecture = max(model_stats, key=lambda n: model_stats[n][0]) if model_stats else '—'

# ── Build report string ──────────────────────────────────────────
W  = 80
sep  = '=' * W
thin = '-' * W
now  = datetime.now().strftime('%Y-%m-%d  %H:%M:%S')

lines = []
lines.append(sep)
lines.append('  putEMG — Within-Subject Feature-Based Evaluation Report')
lines.append(f'  Generated : {now}')
lines.append(sep)
lines.append('')
lines.append('Config')
lines.append(thin)
lines.append(f'  Models tested      : {", ".join(model_registry.keys())}')
lines.append(f'  Subjects evaluated : {len(subject_results)}')
lines.append(f'  Folds per subject  : {N_FOLDS}')
lines.append(f'  Dropout            : {DROPOUT}')
lines.append(f'  Max epochs         : {MAX_EPOCHS}')
lines.append(f'  Early stop         : patience={PATIENCE}, min_delta={MIN_DELTA}')
lines.append(f'  Seed               : {SEED}')
lines.append(f'  Feature mode       : {MODE}  (libemg, 50 features × 24 channels)')
lines.append('')

# ── Model summary ──────────────────────────────────────────────
lines.append(thin)
lines.append('Model Summary  (grand mean ± std across subjects)')
lines.append(thin)
lines.append(f'  {"Model":<22}  {"Subjects":>8}  {"Mean Acc":>10}  {"Std":>8}')
lines.append(f'  {"-"*22}  {"-"*8}  {"-"*10}  {"-"*8}')
for name, (mean, std, n) in sorted(model_stats.items(), key=lambda x: -x[1][0]):
    marker = '  ←' if name == best_architecture else ''
    lines.append(f'  {name:<22}  {n:>8}  {mean:>9.2f}%  {std:>7.2f}%{marker}')
lines.append('')
lines.append(f'  → Best architecture : {best_architecture}  '
             f'({model_stats[best_architecture][0]:.2f}%)')
lines.append(f'  → Weights saved in  : {os.path.join(WEIGHTS_DIR, best_architecture)}/')
lines.append('')

# ── Per-subject breakdown ─────────────────────────────────────────
model_names = list(model_registry.keys())
col = 12

lines.append(thin)
lines.append('Per-Subject Results')
lines.append(thin)
header = f'  {"Subject":<12}' + ''.join(f'{n[:col]:>{col+2}}' for n in model_names)
header += f'  {"Best Model":<20}  {"Best Acc":>8}'
lines.append(header)
lines.append(f'  {"-"*12}' + ''.join(f'  {"-"*col}' for _ in model_names)
             + f'  {"-"*20}  {"-"*8}')

for sub_name, results in sorted(subject_results.items()):
    sub_id = f'subject_{sub_name.split("_")[2]}'
    accs   = {name: np.mean(results[name]) * 100
              for name in model_names if name in results and results[name]}
    if not accs:
        continue
    best_model = max(accs, key=accs.get)
    best_acc   = accs[best_model]
    row = f'  {sub_id:<12}'
    for name in model_names:
        val = f'{accs[name]:.2f}%' if name in accs else '—'
        row += f'  {val:>{col}}'
    row += f'  {best_model:<20}  {best_acc:>7.2f}%'
    lines.append(row)

lines.append('')
lines.append(sep)

report_text = '\n'.join(lines)

with open(report_path, 'w') as f:
    f.write(report_text)

print(report_text)
print(f'\nReport saved → {report_path}')